# SecureSpeak — Notebook A2: PhishTank Full Scoring (Find Real BORDERLINE Cases)

## What Happened With PhiUSIIL
PhiUSIIL gave pp=0.013 mean — the model correctly rejects it as not matching
Bangladeshi phishing patterns. This is actually evidence of Bangladesh-specificity.

## New Strategy
Score ALL 60,685 PhishTank phishing URLs (not just 500).
Within 60,685 URLs, even if only 2-3% score in the borderline range,
that gives 1,200-1,800 genuine BORDERLINE cases.
No new datasets needed. Everything is already on your Drive.

## Output
```
cse498R/model_for_research/notebookA2_phishtank_full/
    phishtank_all_scored.csv     <- all 60K+ URLs with pp scores
    phishtank_borderline.csv     <- pp in [0.30, 0.70]
    phishtank_easy.csv           <- pp > 0.70
    notebookA2_summary.json      <- statistics
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse
from collections import Counter

BASE         = '/content/drive/MyDrive/cse498R/Datasets'
SAVED_MODELS = '/content/drive/MyDrive/cse498R/model_for_research/saved_models'
PHASE2_EXT   = '/content/drive/MyDrive/cse498R/model_for_research/phase2_external'
OUT_DIR      = '/content/drive/MyDrive/cse498R/model_for_research/notebookA2_phishtank_full'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Output dir:', OUT_DIR)
print('='*60)

---
## Step 1 — Load Saved URL Model

In [ ]:
url_model  = joblib.load(f'{SAVED_MODELS}/url_model.joblib')
scaler_url = joblib.load(f'{SAVED_MODELS}/url_scaler.joblib')
url_meta   = json.load(open(f'{SAVED_MODELS}/url_meta.json'))
print(f'Model: {url_meta["best_model_name"]}')
print(f'Internal acc: {url_meta["internal_accuracy"]}')

---
## Step 2 — URL Feature Engineering
Exact copy from Blackbook. Imports included at top to avoid NameError.

In [ ]:
# Required imports for feature engineering
import re, math
from urllib.parse import urlparse
from collections import Counter

try:
    import tldextract; TLD_OK = True
except: TLD_OK = False

HIGH_RISK_TLDS = {'tk','ml','ga','cf','gq','pw','top','xyz','online','site','club',
                  'live','shop','info','biz','link','click','download','stream'}
FREE_HOST_TLDS = {'tk','ml','ga','cf','gq','pw'}
FINANCIAL_KW   = ['bank','login','secure','verify','update','account','password','signin',
                  'bkash','nagad','rocket','paypal','amazon','netflix','microsoft','apple','google','confirm']
BRAND_KW       = ['paypal','amazon','google','facebook','apple','microsoft','bkash','nagad','rocket']

def shannon_entropy(s):
    if not s: return 0.0
    f = {}
    for c in s: f[c] = f.get(c, 0) + 1
    n = len(s)
    return -sum((v/n)*math.log2(v/n) for v in f.values())

def engineer_url_features(url):
    url = str(url).strip().lower()
    if TLD_OK:
        ext = tldextract.extract(url)
        domain, suffix, subdomain = ext.domain, ext.suffix, ext.subdomain
    else:
        m = re.search(r'(?:https?://)?([^/]+)', url)
        host = m.group(1) if m else url
        parts = host.split('.')
        domain    = parts[-2] if len(parts) >= 2 else host
        suffix    = parts[-1] if len(parts) >= 1 else ''
        subdomain = '.'.join(parts[:-2]) if len(parts) > 2 else ''
    path  = re.sub(r'https?://[^/]+', '', url)
    query = path.split('?', 1)[1] if '?' in path else ''
    return [
        min(len(url)/500, 1.0),
        min(url.count('.')/10, 1.0),
        min(url.count('/')/15, 1.0),
        min(len(re.findall(r'[-_@!%&=+]', url))/20, 1.0),
        sum(c.isdigit() for c in url)/max(len(url), 1),
        sum(c.isalpha() for c in url)/max(len(url), 1),
        1.0 if suffix in HIGH_RISK_TLDS else 0.0,
        min(subdomain.count('.')+1 if subdomain else 0, 5)/5,
        min(len(domain)/30, 1.0),
        1.0 if re.match(r'^(?:\d{1,3}\.){3}\d{1,3}$',
                        url.split('/')[2] if '/' in url else url) else 0.0,
        min(sum(b in domain for b in BRAND_KW), 3)/3,
        min(len(path)/200, 1.0),
        min(len([s for s in path.split('/') if s])/10, 1.0),
        1.0 if '?' in url else 0.0,
        min(len(query)/200, 1.0),
        1.0 if url.startswith('https') else 0.0,
        1.0 if 'https' in path else 0.0,
        shannon_entropy(url)/6.0,
        shannon_entropy(domain)/4.0,
        min(sum(kw in url for kw in FINANCIAL_KW), 5)/5,
        1.0 if re.search(r'@|//.*@', url) else 0.0,
        min(url.count('-')/8, 1.0),
        1.0 if len(url) > 75 and not url.startswith('https') else 0.0,
        1.0 if suffix in FREE_HOST_TLDS else 0.0,
        min(len(re.findall(r'\d{3,}', url))/3, 1.0),
        (1.0 if url.startswith('https') else 0.0) * (0.0 if suffix in HIGH_RISK_TLDS else 1.0),
    ]

URL_FEAT_NAMES = [
    'url_length','dot_count','slash_count','special_chars','digit_ratio','letter_ratio',
    'high_risk_tld','subdomain_depth','domain_length','uses_ip','brand_impersonation',
    'path_length','path_segments','has_query','query_length','has_https','https_in_path',
    'url_entropy','domain_entropy','financial_kw','at_in_url','hyphen_count','long_http',
    'free_hosting_tld','long_numbers','https_x_safe_tld',
]
assert len(URL_FEAT_NAMES) == 26
print('URL feature engineer ready: 26 named features.')


---
## Step 3 — Load ALL PhishTank URLs and Score Them

In [ ]:
PHISHTANK_PATH = f'{PHASE2_EXT}/phishtank_raw.csv'
print(f'Loading: {PHISHTANK_PATH}')
df_pt = pd.read_csv(PHISHTANK_PATH, low_memory=False)
pt_url_col = next((c for c in df_pt.columns if c.lower() == 'url'), None)
all_phish_urls = df_pt[pt_url_col].dropna().astype(str).tolist()
print(f'Total PhishTank URLs: {len(all_phish_urls):,}')

# Score in batches of 1000 for memory efficiency
BATCH = 1000
all_pp = []
print(f'Scoring in batches of {BATCH}...')
t0 = time.time()
for i in range(0, len(all_phish_urls), BATCH):
    batch = all_phish_urls[i:i+BATCH]
    X = np.array([engineer_url_features(u) for u in batch], dtype=np.float32)
    X_s = scaler_url.transform(X)
    pp_batch = url_model.predict_proba(X_s)[:, 1].astype(np.float32)
    all_pp.extend(pp_batch.tolist())
    if (i // BATCH) % 5 == 0:
        print(f'  Scored {min(i+BATCH, len(all_phish_urls)):,} / {len(all_phish_urls):,}')

pp_all = np.array(all_pp, dtype=np.float32)
print(f'Done in {time.time()-t0:.1f}s')

print(f'\n=== PP DISTRIBUTION ON ALL {len(pp_all):,} PhishTank URLs ===')
print(f'Mean:   {pp_all.mean():.4f}')
print(f'Std:    {pp_all.std():.4f}')
print(f'Median: {np.median(pp_all):.4f}')
print()
bins = [(0.0,0.30,'Low (<0.30)'),
        (0.30,0.50,'Low-Mid (0.30-0.50) BORDERLINE'),
        (0.50,0.70,'Mid-High (0.50-0.70) BORDERLINE'),
        (0.70,1.01,'High (>0.70) easy phishing')]
for lo, hi, label in bins:
    count = ((pp_all >= lo) & (pp_all < hi)).sum()
    pct = count / len(pp_all) * 100
    print(f'  [{lo:.2f}-{hi:.2f}]  {label:<35} {count:>6,} ({pct:.1f}%)')

---
## Step 4 — Extract BORDERLINE Subset

In [ ]:
df_scored = pd.DataFrame({
    'url': all_phish_urls,
    'pp':  pp_all,
    'true_label': 1,
    'source': 'phishtank',
})

BORDER_LO, BORDER_HI = 0.30, 0.70
df_border  = df_scored[(df_scored.pp >= BORDER_LO) & (df_scored.pp <= BORDER_HI)].copy()
df_easy    = df_scored[df_scored.pp > BORDER_HI].copy()
df_missed  = df_scored[df_scored.pp < BORDER_LO].copy()

print(f'BORDERLINE (pp 0.30-0.70): {len(df_border):,}')
print(f'Easy phishing (pp>0.70):   {len(df_easy):,}')
print(f'Missed (pp<0.30):          {len(df_missed):,}')

if len(df_border) > 0:
    print(f'\nBORDERLINE pp stats:')
    print(f'  mean={df_border.pp.mean():.3f}  std={df_border.pp.std():.3f}')
    print(f'\nSample BORDERLINE URLs:')
    for u, p in df_border[['url','pp']].head(8).values:
        print(f'  pp={p:.3f}  {str(u)[:90]}')

---
## Step 5 — Plot and Save

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(pp_all, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(BORDER_LO, color='orange', linestyle='--', linewidth=2,
           label=f'BORDERLINE lower ({BORDER_LO})')
ax.axvline(BORDER_HI, color='red', linestyle='--', linewidth=2,
           label=f'BORDERLINE upper ({BORDER_HI})')
ax.set_title(f'PP Score Distribution — All {len(pp_all):,} PhishTank URLs\n'
              'BORDERLINE region highlighted', fontweight='bold')
ax.set_xlabel('PP Score')
ax.set_ylabel('Count')
ax.legend()
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/phishtank_full_pp_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# Save files
df_scored.to_csv(f'{OUT_DIR}/phishtank_all_scored.csv', index=False)
df_border.to_csv(f'{OUT_DIR}/phishtank_borderline.csv', index=False)
df_easy.to_csv(f'{OUT_DIR}/phishtank_easy.csv', index=False)

summary = {
    'generated_at': datetime.now().isoformat(),
    'total_phishtank_urls': int(len(df_scored)),
    'borderline_count': int(len(df_border)),
    'easy_phish_count': int(len(df_easy)),
    'missed_count': int(len(df_missed)),
    'borderline_range': [BORDER_LO, BORDER_HI],
    'pp_stats': {
        'mean':   float(pp_all.mean()),
        'std':    float(pp_all.std()),
        'median': float(np.median(pp_all)),
        'min':    float(pp_all.min()),
        'max':    float(pp_all.max()),
    },
    'phiusiil_finding': (
        'PhiUSIIL mean pp=0.013 confirms model is Bangladesh-specific. '
        'PhishTank is the correct external source for this system.'
    ),
    'verdict': (
        f'BORDERLINE cases found: {len(df_border):,} — proceed to Notebook B'
        if len(df_border) >= 100
        else f'Only {len(df_border)} BORDERLINE cases — discuss with Claude'
    )
}
with open(f'{OUT_DIR}/notebookA2_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('='*60)
print('NOTEBOOK A2 COMPLETE')
print('='*60)
print(f'BORDERLINE count: {len(df_border):,}')
print(f'Verdict: {summary["verdict"]}')
print(f'\nSend notebookA2_summary.json to confirm.')